# Session 14 · Decision Trees: Rules a Human Can Read

Module 3 gave us classifiers that *work* but can't easily *explain themselves*.
Today's model can: a **decision tree** decides by asking yes/no questions, and you
can read the whole thing aloud as if-then rules.

We'll build the first split **by hand** — counting pass/fail on each side of a cut —
then let sklearn confirm it picks the very same line.

> ✏️ = your cell to fill in. Gaps never block the run; the notebook works untouched.

## Step 1 · Meet 12 students we already know

We predict `passed` from **habit** features (study hours, attendance, sleep, screen
time, practice) — **not** from `test_score`, which is basically the label in disguise
(it's where pass/fail comes from). A tree given `test_score` would learn nothing about habits.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

full = pd.read_csv("../../../datasets/anchor/student_habits.csv")
habits = ["study_hours_per_week","attendance_pct","sleep_hours_per_night","screen_time_hours_per_day","practice_sessions_per_week"]

# a hand-picked sample of 12 students for the by-hand exercise
sample_ids = ["S169","S303","S221","S343","S352","S024","S347","S312","S172","S062","S257","S237"]
sample = (full[full['student_id'].isin(sample_ids)]
          [['student_id','study_hours_per_week','attendance_pct','passed']]
          .sort_values('study_hours_per_week')
          .reset_index(drop=True))
print('parent pile:', int(sample['passed'].sum()), 'passed /',
      int((1-sample['passed']).sum()), 'failed  (as mixed as it gets)')
sample

## Step 2 · Try three cuts on study hours — by counting

A tree split is just this: pick a threshold, send students left (below) or right
(at/above), and count pass/fail on each side. The best cut makes each side as
**one-colour** as possible. Run the helper, then read the counts.

In [ ]:
def count_split(df, col, thr):
    left  = df[df[col] <  thr]
    right = df[df[col] >= thr]
    lp, lf = int(left['passed'].sum()),  int((1-left['passed']).sum())
    rp, rf = int(right['passed'].sum()), int((1-right['passed']).sum())
    print(f'{col} < {thr}:')
    print(f'   LEFT  (below): {len(left):2d} students  ->  {lp} pass / {lf} fail')
    print(f'   RIGHT (at/above): {len(right):2d} students  ->  {rp} pass / {rf} fail')
    print()

for thr in [5, 8, 12]:
    count_split(sample, 'study_hours_per_week', thr)

**✏️ Which cut is cleanest?** Look at the three splits above and answer in the
cell below (just replace the `...`). Which threshold makes the two sides most
one-colour — i.e. closest to *all fail on the left, all pass on the right*?

In [ ]:
# ✏️ your answer (a number: 5, 8, or 12)
best_cut = ...   # e.g. best_cut = 8
print('I think the cleanest split is study_hours <', best_cut)

## Step 3 · Put a number on 'how mixed' — impurity (Gini)

"Cleaner" has a proper name: **purity**. Its opposite, **impurity**, has a simple
number (Gini): **0** = all one class (pure), **0.5** = 50/50 (as mixed as possible).
A tree picks the split that drops impurity the most. You don't memorise the formula —
you read the number as *how muddy is this pile*.

In [ ]:
def gini(df):
    n = len(df)
    if n == 0:
        return 0.0
    p = df['passed'].mean()
    return 1 - p**2 - (1-p)**2

def weighted_gini(df, col, thr):
    left, right = df[df[col] < thr], df[df[col] >= thr]
    n = len(df)
    return (len(left)*gini(left) + len(right)*gini(right)) / n

print(f'parent pile impurity: {gini(sample):.3f}   (6 pass / 6 fail = as mixed as it gets)')
print()
for thr in [5, 8, 12]:
    print(f'study < {thr:2d}  ->  weighted impurity after split = {weighted_gini(sample, "study_hours_per_week", thr):.3f}')
print()
print('Lowest impurity = cleanest split. study < 8 wins (0.143): one pure fail pile,')
print('one almost-pure pass pile. The number just confirms the counting.')

## Step 4 · Let sklearn confirm the root — on the *full* data

Now the reveal. `DecisionTreeClassifier` scans every feature and every threshold,
counts exactly like we did, and keeps the best. With `max_depth=1` it makes just the
**root split**. Watch which question it picks.

In [ ]:
X = full[habits]
y = full['passed']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

stump = DecisionTreeClassifier(max_depth=1, random_state=0)
stump.fit(X_train, y_train)

print(export_text(stump, feature_names=habits))
print('train accuracy:', round(stump.score(X_train, y_train), 3))
print('test  accuracy:', round(stump.score(X_test, y_test), 3))
print()
print('Root split: study_hours_per_week <= 7.65 — essentially the study < 8 line')
print('we drew by hand. The algorithm IS the counting, done faster.')

## Step 5 · Grow to depth 2 and read the tree *aloud*

Add one more level of questions. Then trace each path top-to-bottom and turn it
into a plain-English rule — that sentence *is* the model.

In [ ]:
tree2 = DecisionTreeClassifier(max_depth=2, random_state=0)
tree2.fit(X_train, y_train)
print(export_text(tree2, feature_names=habits))

fig, ax = plt.subplots(figsize=(11, 5))
plot_tree(tree2, feature_names=habits, class_names=['fail','pass'],
          filled=True, impurity=True, ax=ax, fontsize=9)
ax.set_title('A student pass/fail tree you can read aloud')
plt.tight_layout()
plt.show()

**✏️ Read one path aloud.** Pick any path from the top of the tree to a leaf and
write it as an if-then sentence, e.g. *'If study hours are below 7.65, predict FAIL.'*

In [ ]:
# ✏️ replace this string with your read-aloud rule for one path
my_rule = '...'
print(my_rule)

## Step 6 · ✏️ The one it gets wrong — a hint of trouble

Our clean split sent the 'study >= 8' group to *pass*, but one student there actually
**failed** (they studied plenty and still didn't pass). The shallow tree gets them
wrong — and that's OK. A good model is *mostly* right, not *perfectly* right.

Below we find that student. Then answer the reflection — it seeds the Session 16
overfitting deep-dive.

In [ ]:
# the sample student(s) our 'study >= 8 -> pass' rule misclassifies
right_side = sample[sample['study_hours_per_week'] >= 8]
exceptions = right_side[right_side['passed'] == 0]
print('Studied 8+ hours but still FAILED:')
exceptions

**✏️ Your reflection (2–3 sentences).** We *could* keep adding questions until the
tree gets even this student right — every last exception. Why might a tree that gets
**every** training student correct actually be *worse* at predicting new students?

*(Write your answer here, replacing this line.)*

## Wrap-up

- A **decision tree** decides by asking yes/no questions — a model you can **read aloud**.
- It learns by **counting**: pick the split that makes the two sides purest (lowest impurity).
- By hand we found *study < 8*; sklearn confirmed *study <= 7.65*. Same line, faster.
- The one student our clean split misses is honest — and a hint that chasing *zero*
  errors means **memorising**.

**Next (S15):** grow the tree and ask it *which habit mattered most* — feature importance.